# Differential Attention: Low-SNR-trained vs All-SNR-trained

This notebook trains a differential-attention model only on low SNR:

```text
-20 dB to 0 dB
```

Then it compares that model against the existing all-SNR-trained differential-attention model.

Fair comparison rule:

- The low-SNR model is trained on `-20..0 dB` only.
- The all-SNR model is loaded from `experiments/5class_diffattention`.
- Both models are evaluated on the same low-SNR test split.
- Final zip exports only `experiments/5class_diffattention_lowsnr/...`.


In [ ]:
# CELL 1: Setup repo and paths
import os
import sys
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

os.environ["KERAS_BACKEND"] = "tensorflow"

REPO_URL = "https://github.com/akshlabh/amr-5-class.git"
WORK_DIR = Path("/kaggle/working/amr-5-class")
DATASET = Path("/kaggle/input/datasets/gustavopolicarpo/rml201610a-dict/RML2016.10a_dict.dat")


def find_attached_repo():
    for root in Path("/kaggle/input").glob("**"):
        if (root / "src" / "train.py").exists() and (root / "configs").exists():
            return root
    return None


if (WORK_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(WORK_DIR), "pull"], check=False)
elif WORK_DIR.exists() and (WORK_DIR / "src" / "train.py").exists():
    print(f"Using existing work dir: {WORK_DIR}")
else:
    attached = find_attached_repo()
    if attached is not None:
        print(f"Copying attached repo from: {attached}")
        if WORK_DIR.exists():
            shutil.rmtree(WORK_DIR)
        shutil.copytree(attached, WORK_DIR)
    else:
        print("Cloning repo from GitHub...")
        subprocess.run(["git", "clone", REPO_URL, str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

ALLSNR_DIR = Path("experiments/5class_diffattention")
LOWSNR_DIR = Path("experiments/5class_diffattention_lowsnr")
COMPARE_DIR = LOWSNR_DIR / "comparison_with_all_snr_diffattention"
MODEL_VARIANT = "diffattention_trained_only_minus20_to_0db_standard_loss"
VARIANT_FILE = Path("results/model_variant.txt")

required = [
    "src/models/mcldnn_diffattention.py",
    "configs/exp_5class_diffattention.yaml",
    "configs/exp_5class_diffattention_lowsnr.yaml",
    "src/train.py",
]

print(f"Working dir: {Path.cwd()}")
print(f"Dataset    : {DATASET}")
print(f"Dataset OK : {DATASET.exists()}")
for f in required:
    print(f"{'OK' if Path(f).exists() else 'MISSING'} {f}")

assert DATASET.exists(), f"Dataset not found: {DATASET}"
for f in required:
    assert Path(f).exists(), f"Required repo file missing: {f}"


In [ ]:
# CELL 2: Quick model shape check
import gc
import keras
import keras.backend as K

keras.mixed_precision.set_global_policy("float32")
K.clear_session(); gc.collect()

from src.models.mcldnn_diffattention import (
    build_mcldnn_diffattention,
    build_mcldnn_diffattention_extractor,
)

model = build_mcldnn_diffattention(classes=5)
extractor = build_mcldnn_diffattention_extractor(classes=5)

print(f"Diff-attention params: {model.count_params():,}")
print(f"Under 300k          : {'YES' if model.count_params() < 300_000 else 'NO'}")

x1 = np.zeros((4, 2, 128, 1), dtype="float32")
x2 = np.zeros((4, 128, 1), dtype="float32")
x3 = np.zeros((4, 128, 1), dtype="float32")

pred = model.predict([x1, x2, x3], verbose=0)
pred2, attn = extractor.predict([x1, x2, x3], verbose=0)

print(f"Training output shape : {pred.shape}")
print(f"Extractor output shape: {pred2.shape}")
print(f"Attention map shape   : {attn.shape}")

assert pred.shape == (4, 5)
assert pred2.shape == (4, 5)
assert attn.shape == (4, 2, 124, 124)
assert model.count_params() < 300_000
print("Shape checks passed")

del model, extractor
K.clear_session(); gc.collect()


In [ ]:
# CELL 3: Train low-SNR-only differential attention model from scratch
# Existing low-SNR outputs are removed first so stale weights are not reused.

if LOWSNR_DIR.exists():
    print(f"Removing existing low-SNR diff-attention outputs: {LOWSNR_DIR}")
    shutil.rmtree(LOWSNR_DIR)

print("Training differential attention on -20 dB to 0 dB only...\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
process = subprocess.Popen(
    [
        sys.executable, "-u", "src/train.py",
        "--config", "configs/exp_5class_diffattention_lowsnr.yaml",
        "--datasetpath", str(DATASET),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)

(LOWSNR_DIR / "results").mkdir(parents=True, exist_ok=True)
(LOWSNR_DIR / VARIANT_FILE).write_text(MODEL_VARIANT)

print("\nLow-SNR diff-attention training finished.")
print("Checkpoint:", (LOWSNR_DIR / "checkpoints" / "best_model.weights.h5").exists())
print("Test score:", (LOWSNR_DIR / "results" / "test_score.csv").exists())
print("SNR CSV   :", (LOWSNR_DIR / "results" / "acc_per_snr.csv").exists())
print("Variant   :", (LOWSNR_DIR / VARIANT_FILE).read_text().strip())


In [ ]:
# CELL 4: Ensure all-SNR-trained diff-attention checkpoint exists
# This is used only for comparison, not exported in the low-SNR zip.

ALL_WEIGHTS = ALLSNR_DIR / "checkpoints" / "best_model.weights.h5"
ALL_SCORE = ALLSNR_DIR / "results" / "test_score.csv"
ALL_SNR = ALLSNR_DIR / "results" / "acc_per_snr.csv"

if ALL_WEIGHTS.exists() and ALL_SCORE.exists() and ALL_SNR.exists():
    print("All-SNR-trained diff-attention results already exist. Using for comparison.")
else:
    print("All-SNR diff-attention checkpoint/results missing. Training all-SNR model for comparison...")
    subprocess.run([
        sys.executable, "src/train.py",
        "--config", "configs/exp_5class_diffattention.yaml",
        "--datasetpath", str(DATASET),
    ], check=True)

print("All-SNR weights:", ALL_WEIGHTS.exists())
print("All-SNR score  :", ALL_SCORE.exists())
print("All-SNR SNR CSV:", ALL_SNR.exists())


In [ ]:
# CELL 5: Fair evaluation of both models on the same -20..0 dB test split
from src.dataset import load_data, FIVE_CLASS
from src.models.mcldnn_diffattention import build_mcldnn_diffattention
from src.utils.mltools import calculate_confusion_matrix, plot_confusion_matrix

K.clear_session(); gc.collect()
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

(mods, snrs, lbl), _, _, (X_test, Y_test), (_, _, test_idx) = load_data(
    str(DATASET), FIVE_CLASS, seed=2016, snr_range=(-20, 0), shuffle_split=False
)
test_SNRs = np.array([lbl[i][1] for i in test_idx])
print(f"Low-SNR test set: {X_test.shape[0]} samples, SNRs={snrs}")


def make_inputs(X):
    return [
        np.expand_dims(X, axis=3).astype("float32"),
        np.expand_dims(X[:, 0, :], axis=2).astype("float32"),
        np.expand_dims(X[:, 1, :], axis=2).astype("float32"),
    ]


def ce_loss(Y, P, eps=1e-7):
    P = np.clip(P, eps, 1.0 - eps)
    return float(-np.mean(np.sum(Y * np.log(P), axis=1)))


def acc(Y, P):
    return float(np.mean(np.argmax(Y, axis=1) == np.argmax(P, axis=1)))

inputs_test = make_inputs(X_test)

all_model = build_mcldnn_diffattention(classes=5)
all_model.load_weights(ALLSNR_DIR / "checkpoints" / "best_model.weights.h5")
print("Predicting all-SNR-trained diff-attention on low-SNR test set...")
all_pred = all_model.predict(inputs_test, batch_size=400, verbose=1)
del all_model; K.clear_session(); gc.collect()

low_model = build_mcldnn_diffattention(classes=5)
low_model.load_weights(LOWSNR_DIR / "checkpoints" / "best_model.weights.h5")
print("Predicting low-SNR-trained diff-attention on low-SNR test set...")
low_pred = low_model.predict(inputs_test, batch_size=400, verbose=1)
del low_model; K.clear_session(); gc.collect()

summary = pd.DataFrame([
    {
        "model": "diffattention_trained_all_snrs",
        "eval_set": "test_-20_to_0db",
        "loss": ce_loss(Y_test, all_pred),
        "accuracy": acc(Y_test, all_pred),
    },
    {
        "model": "diffattention_trained_lowsnr_only",
        "eval_set": "test_-20_to_0db",
        "loss": ce_loss(Y_test, low_pred),
        "accuracy": acc(Y_test, low_pred),
    },
])
summary["accuracy_percent"] = 100.0 * summary["accuracy"]
summary_csv = COMPARE_DIR / "all_snr_vs_lowsnr_diffattention_summary_on_lowsnr_test.csv"
summary.to_csv(summary_csv, index=False)
print("Summary on same low-SNR test split:")
display(summary)

rows = []
acc_all = {}
acc_low = {}
acc_mod_snr_low = np.zeros((len(mods), len(snrs)))
for i, snr in enumerate(snrs):
    mask = test_SNRs == snr
    Ya = Y_test[mask]
    Pa = all_pred[mask]
    Pl = low_pred[mask]
    conf_low, cor, ncor = calculate_confusion_matrix(Ya, Pl, mods)
    acc_all[snr] = acc(Ya, Pa)
    acc_low[snr] = acc(Ya, Pl)
    acc_mod_snr_low[:, i] = np.round(np.diag(conf_low) / np.sum(conf_low, axis=1), 4)
    rows.append({
        "snr": snr,
        "all_snr_trained_acc": acc_all[snr],
        "lowsnr_trained_acc": acc_low[snr],
        "delta_lowsnr_minus_all": acc_low[snr] - acc_all[snr],
        "all_snr_trained_acc_percent": 100.0 * acc_all[snr],
        "lowsnr_trained_acc_percent": 100.0 * acc_low[snr],
        "delta_percent_points": 100.0 * (acc_low[snr] - acc_all[snr]),
    })

comparison = pd.DataFrame(rows)
comparison_csv = COMPARE_DIR / "all_snr_vs_lowsnr_diffattention_acc_per_snr.csv"
comparison.to_csv(comparison_csv, index=False)
print("Per-SNR comparison on same low-SNR test split:")
display(comparison)

conf_all, _, _ = calculate_confusion_matrix(Y_test, all_pred, mods)
conf_low, _, _ = calculate_confusion_matrix(Y_test, low_pred, mods)
conf_delta = conf_low - conf_all
np.savetxt(COMPARE_DIR / "confusion_all_snr_trained_on_lowsnr_test.csv", conf_all, delimiter=",")
np.savetxt(COMPARE_DIR / "confusion_lowsnr_trained_on_lowsnr_test.csv", conf_low, delimiter=",")
np.savetxt(COMPARE_DIR / "confusion_lowsnr_minus_all_snr_on_lowsnr_test.csv", conf_delta, delimiter=",")

plot_confusion_matrix(
    conf_all, labels=mods,
    title="All-SNR trained diff-attention on -20..0 dB test",
    save_filename=str(COMPARE_DIR / "confusion_all_snr_trained_on_lowsnr_test.png"),
)
plot_confusion_matrix(
    conf_low, labels=mods,
    title="Low-SNR trained diff-attention on -20..0 dB test",
    save_filename=str(COMPARE_DIR / "confusion_lowsnr_trained_on_lowsnr_test.png"),
)

print(f"Saved: {summary_csv}")
print(f"Saved: {comparison_csv}")


In [ ]:
# CELL 6: Plot comparison curves and confusion difference
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(comparison["snr"], comparison["all_snr_trained_acc_percent"], marker="o", linewidth=2.4, label="Diff-attention trained on all SNRs")
ax.plot(comparison["snr"], comparison["lowsnr_trained_acc_percent"], marker="s", linewidth=2.4, label="Diff-attention trained only on -20..0 dB")
ax.set_title("Differential Attention: Low-SNR-trained vs All-SNR-trained")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Accuracy on same low-SNR test split (%)")
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_xticks(comparison["snr"])
plt.tight_layout()
curve_path = COMPARE_DIR / "all_snr_vs_lowsnr_diffattention_acc_vs_snr.png"
fig.savefig(curve_path, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 4.8))
colors = ["#2ca02c" if x >= 0 else "#d62728" for x in comparison["delta_percent_points"]]
ax.bar(comparison["snr"].astype(str), comparison["delta_percent_points"], color=colors)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Low-SNR-trained minus All-SNR-trained by SNR")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Delta accuracy (percentage points)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
delta_path = COMPARE_DIR / "lowsnr_minus_all_snr_delta_by_snr.png"
fig.savefig(delta_path, dpi=180, bbox_inches="tight")
plt.show()

classes = mods
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, cm, title in [
    (axes[0], conf_all, "All-SNR trained"),
    (axes[1], conf_low, "Low-SNR trained"),
]:
    im = ax.imshow(cm * 100, vmin=0, vmax=100, cmap="Blues")
    ax.set_title(title)
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=45, ha="right")
    ax.set_yticklabels(classes)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, f"{cm[i,j]*100:.0f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="Row-normalized confusion (%)")
plt.tight_layout()
conf_compare_path = COMPARE_DIR / "all_snr_vs_lowsnr_confusion_on_lowsnr_test.png"
fig.savefig(conf_compare_path, dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
lim = max(1e-6, abs(conf_delta * 100).max())
im = ax.imshow(conf_delta * 100, cmap="RdYlGn", vmin=-lim, vmax=lim)
ax.set_title("Low-SNR-trained minus All-SNR-trained confusion difference")
ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes, rotation=45, ha="right")
ax.set_yticklabels(classes)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, f"{conf_delta[i,j]*100:+.1f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.85, label="Difference (percentage points)")
plt.tight_layout()
conf_delta_path = COMPARE_DIR / "lowsnr_minus_all_snr_confusion_delta.png"
fig.savefig(conf_delta_path, dpi=180, bbox_inches="tight")
plt.show()

print(f"Saved: {curve_path}")
print(f"Saved: {delta_path}")
print(f"Saved: {conf_compare_path}")
print(f"Saved: {conf_delta_path}")


In [ ]:
# CELL 7: Copy comparison outputs into low-SNR result folder
LOWSNR_COMPARE_EXPORT = LOWSNR_DIR / "results" / "comparison_with_all_snr_diffattention"
LOWSNR_COMPARE_EXPORT.mkdir(parents=True, exist_ok=True)

for src in sorted(COMPARE_DIR.glob("*")):
    if src.is_file():
        dst = LOWSNR_COMPARE_EXPORT / src.name
        shutil.copy2(src, dst)
        print(f"Copied: {dst}")


In [ ]:
# CELL 8: Create repo-ready zip for ONLY low-SNR diff-attention results
stamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_base = Path("/kaggle/working") / f"diffattention_lowsnr_repo_ready_{stamp}"
zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(WORK_DIR),
    base_dir="experiments/5class_diffattention_lowsnr",
)

print(f"Created repo-ready zip: {zip_path}")
print("\nExtract this zip at repo root. It will create/update:")
print("  experiments/5class_diffattention_lowsnr/")
print("\nIncluded files:")
for path in sorted(LOWSNR_DIR.rglob("*")):
    if path.is_file():
        print(" -", Path("experiments/5class_diffattention_lowsnr") / path.relative_to(LOWSNR_DIR))

display(FileLink(zip_path))
